# B2-023 — Practice p07

*50 minutes.*

**Program layer:** Round 2 extension  
**Compute:** `compute.policy: cpu` · seed `20261015`  
**Qualified prerequisites:** `book1:F1-scientific-python`, `book1:F3-matrices`, `book1:F4-multivar-calculus`, `book1:F5-probability`, `book1:C1-ml-fundamentals`, `book1:C2-linear-models`, `book1:C5-neural-networks`, `book1:C6-pytorch`, `book1:C11-neural-training`, `B2-019-attention-transformers`, `B2-020-language-transformers`, `B2-021-cross-modal-transformers-vision`, `B2-022-probabilistic-latent-models`  
**Remediation links actually used:** [book1:F1-scientific-python](../../../../book1/units/F1-scientific-python/lesson.ipynb), [book1:F3-matrices](../../../../book1/units/F3-matrices/lesson.ipynb), [book1:F4-multivar-calculus](../../../../book1/units/F4-multivar-calculus/lesson.ipynb), [book1:F5-probability](../../../../book1/units/F5-probability/lesson.ipynb), [book1:C1-ml-fundamentals](../../../../book1/units/C1-ml-fundamentals/lesson.ipynb), [book1:C2-linear-models](../../../../book1/units/C2-linear-models/lesson.ipynb), [book1:C5-neural-networks](../../../../book1/units/C5-neural-networks/lesson.ipynb), [book1:C6-pytorch](../../../../book1/units/C6-pytorch/lesson.ipynb), [book1:C11-neural-training](../../../../book1/units/C11-neural-training/lesson.ipynb), [B2-019-attention-transformers](../../B2-019-attention-transformers/lesson.ipynb), [B2-020-language-transformers](../../B2-020-language-transformers/lesson.ipynb), [B2-021-cross-modal-transformers-vision](../../B2-021-cross-modal-transformers-vision/lesson.ipynb), [B2-022-probabilistic-latent-models](../../B2-022-probabilistic-latent-models/lesson.ipynb).

**Data boundary:** CPU only; use literal unit data, the seeded unit generator, and standard library/NumPy/Torch only. Do not use the web, downloads, external datasets, torchvision datasets, model hubs, pretrained models, checkpoints, or student data.

**Set:** B · **Type:** constrained-coding · **Difficulty:** core · **Time budget:** 50 minutes  
**Concepts:** generative-adversarial-network

## Task

Implement one alternating GAN update through pinned helpers, and prove by probes which network each half changes.

**Contract.**
`discriminator_loss(real_logits, fake_logits)` returns `F.binary_cross_entropy_with_logits(real_logits, torch.ones_like(real_logits)) + F.binary_cross_entropy_with_logits(fake_logits, torch.zeros_like(fake_logits))` (sum of two batch means) and `generator_loss(fake_logits)` returns `F.binary_cross_entropy_with_logits(fake_logits, torch.ones_like(fake_logits))` (non-saturating), both with the p06 contract.
`d_step(G, D, opt_d, real, z)` runs `opt_d.zero_grad(set_to_none=True)`, computes `real_logits = D(real)` and then `fake_logits = D(G(z).detach())`, forms `loss = discriminator_loss(real_logits, fake_logits)`, calls `loss.backward()` and `opt_d.step()`, and returns `loss.item()`.
`g_step(G, D, opt_g, z)` runs `opt_g.zero_grad(set_to_none=True)`, forms `loss = generator_loss(D(G(z)))`, calls `loss.backward()` and `opt_g.step()`, and returns `loss.item()`.
`gan_step(G, D, opt_g, opt_d, real, z)` calls `d_step(G, D, opt_d, real, z)` and then `g_step(G, D, opt_g, z)`, and returns the pair `(d_loss, g_loss)` of Python floats.
`make_generator()` returns `nn.Sequential(nn.Linear(2, 32), nn.ReLU(), nn.Linear(32, 32), nn.ReLU(), nn.Linear(32, 2))` and `make_discriminator()` returns `nn.Sequential(nn.Linear(2, 32), nn.ReLU(), nn.Linear(32, 32), nn.ReLU(), nn.Linear(32, 1))` (default initialization; the discriminator outputs logits and has no final activation).

**Banned (zero points if used):** one optimizer over both networks; `torch.no_grad()` in place of `.detach()` inside `d_step`; drawing random numbers inside any helper; `retain_graph=True`.

**Setup.** Call `torch.manual_seed(SEED)`, then `G = make_generator()`, then `D = make_discriminator()`; build `opt_g = torch.optim.Adam(G.parameters(), lr=5e-4, betas=(0.9, 0.999), eps=1e-8, weight_decay=0)` and `opt_d = torch.optim.Adam(D.parameters(), lr=2e-3, betas=(0.9, 0.999), eps=1e-8, weight_decay=0)`. Use the literal `REAL` rows and `Z = torch.randn(4, 2, generator=torch.Generator().manual_seed(SEED))`. Use `copy.deepcopy` for parameter snapshots and for independent copies of the setup.

**Probes.**

1. **D-step.** Snapshot both networks, then call `d_step(G, D, opt_d, REAL, Z)` alone. Assert: every generator parameter is `torch.equal` to its snapshot; **every generator parameter's `.grad` is `None`**; at least one discriminator parameter changed; and the returned float equals `discriminator_loss(D_before(REAL), D_before(G(Z)))` evaluated on the pre-step discriminator snapshot (`abs_tol = 1e-6`).
2. **G-step.** Snapshot again, then call `g_step(G, D, opt_g, Z)`. Assert: every discriminator parameter is `torch.equal` to its snapshot; at least one generator parameter changed; and the returned float equals `generator_loss(D(G_before(Z)))` on the pre-step generator snapshot (`abs_tol = 1e-6`).
3. **Composition.** Rebuild the pinned setup twice from `torch.manual_seed(SEED)`. On the first copy call `gan_step(G1, D1, opt_g1, opt_d1, REAL, Z)`; on the second call `d_step` then `g_step` by hand. All parameters of the two copies agree with `torch.equal`, and the returned pairs agree. On the pinned setup the first `gan_step` returns approximately `(1.4075340033, 0.6521407962)` (`abs_tol = 1e-5`), and both entries are Python floats.
4. **Missing-detach variant.** Write a local variant `d_step_no_detach` that uses `D(G(z))` for the fake logits. On a fresh copy of the setup, show that after it at least one generator parameter's `.grad` is not `None`, while every generator **weight** is still unchanged. Explain in two sentences why only the `.grad` probe can detect this defect.

**Required answer-check assertions.** Probes 1–4.

In [ ]:
import copy
import math
import torch
from torch import nn
import torch.nn.functional as F

SEED = 20261015
REAL = torch.tensor([[1.0, 1.0], [-1.0, 1.0], [-1.0, -1.0], [1.0, -1.0]])
torch.set_num_threads(1)

In [ ]:
# Write the required implementation, probes, and answer-check assertions here.

## Your response

Show every requested derivation, implementation, shape/dtype probe, training certificate, or audit. Use only the permitted literal data and the seeded unit generator, keep train and held-out roles separate, and end coding work with an answer-check section of executable assertions.